# Insurance 05 — PDF evidence checks plus exhaustive SQL through the API

## Business question
Can a reviewer see the dossier's documentary inconsistencies, the linked Excel row and a correctly scoped portfolio total in one application?

This is a **bounded documentary/analytical workflow**, not a free-form RAG or an LLM-to-SQL agent. The questions are explicit application actions. No model or confidential source is used. Coverage always remains NOT_ASSESSED.

Preparation runs offline: XLSX validation, PDF extraction, case checks, SQL database creation and source fingerprints. Requests use only the prepared cache and parameterized read-only SQL. Public insurer references are not silently applied to synthetic policies.

In [1]:
from pathlib import Path
import sys, tempfile, os
from unittest.mock import patch
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'backend/app').is_dir())
if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))
import pandas as pd
from IPython.display import display
from fastapi.testclient import TestClient
from app.main import app
from ingestion.insurance_excel import export_excel
from ingestion.insurance_dossiers import generate_dossiers
from ingestion.prepare_insurance_runtime import prepare_runtime
print('This integration experiment uses 20 rows; notebook 04 separately validates 100,000.')

C:\Users\choun\Downloads\Prudential_Evidence_Lab_MVP_Source\prudential_evidence_lab\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


This integration experiment uses 20 rows; notebook 04 separately validates 100,000.


## 1. Prepare actual XLSX/PDF inputs and exercise the real API
A temporary dataset keeps the experiment fast and independent of the installed local cache. TestClient executes real FastAPI routes; database queries, XLSX reading and PDF extraction are not mocked. Only the configured cache directory changes.

Expected results are declared independently below. A missing invoice is incomplete; a wrong policy, duplicate invoice, amount mismatch or out-of-period date is a conflict. None of these statuses grants or denies coverage.

In [2]:
expected = {
    'D001_consistent': 'READY_FOR_REVIEW',
    'D002_missing_invoice': 'INCOMPLETE',
    'D003_wrong_policy': 'CONFLICT',
    'D004_duplicate_invoice': 'CONFLICT',
    'D005_amount_conflict': 'CONFLICT',
    'D006_outside_period': 'CONFLICT',
}
reports = []
with tempfile.TemporaryDirectory(prefix='insurance_integration_') as tmp:
    root = Path(tmp)
    export_excel(root / 'portfolio', 20)
    generate_dossiers(root / 'dossiers')
    preparation = prepare_runtime(root / 'portfolio', root / 'dossiers', root / 'runtime')
    with patch.dict(os.environ, {'INSURANCE_RUNTIME_DIR': str(root / 'runtime')}):
        with TestClient(app) as client:
            listing = client.get('/api/claims-analytics/cases')
            assert listing.status_code == 200 and listing.json()['rows'] == 20
            for case_id, status in expected.items():
                response = client.get('/api/claims-analytics/cases/' + case_id)
                assert response.status_code == 200
                result = response.json()
                assert result['case']['status'] == status
                assert result['coverage_decision'] == 'NOT_ASSESSED'
                reports.append(result)
            # Simulate corruption only in this throwaway cache.
            with (root / 'runtime/portfolio.sqlite').open('ab') as stream:
                stream.write(b'changed')
            rejection = client.get('/api/claims-analytics/cases').status_code
            assert rejection == 503
display(preparation)
display(pd.DataFrame([{'case': r['case']['id'], 'status': r['case']['status'],
                      'issues': r['case']['issues'], 'coverage': r['coverage_decision'],
                      'excel_row': r['ledger']['source_row']} for r in reports]))
print('Changed database rejected:', rejection)

{'rows': 20,
 'dossiers': 6,
 'output': 'C:\\Users\\choun\\AppData\\Local\\Temp\\insurance_integration_1w2m7o_g\\runtime'}

,case,status,issues,coverage,excel_row
0,D001_consistent,READY_FOR_REVIEW,[],NOT_ASSESSED,2
1,D002_missing_invoice,INCOMPLETE,[missing_invoice],NOT_ASSESSED,3
2,D003_wrong_policy,CONFLICT,[policy_mismatch],NOT_ASSESSED,4
3,D004_duplicate_invoice,CONFLICT,[duplicate_invoice],NOT_ASSESSED,5
4,D005_amount_conflict,CONFLICT,[amount_mismatch],NOT_ASSESSED,6
5,D006_outside_period,CONFLICT,[loss_outside_period],NOT_ASSESSED,7


Changed database rejected: 503


## 2. Verify the SQL population, not just a fluent answer
For the first case, the selected scope is SYNTHETIC_A + water. In the 20-row generator, source indices 0, 4 and 12 satisfy that condition. Their claimed values are 15,025,000 + 56,676 + 120,028 EUR cents; only index 4 has a paid value (39,673 cents).

This hand-calculated small population gives an independent expected result. API parameters come from the selected case's metadata matched against its ledger record, not from arbitrary client SQL. This is a scope consistency control, **not authenticated tenant isolation**.

In [3]:
first = reports[0]
assert first['scope_totals'] == {
    'count': 3, 'claimed_cents': 15_025_000 + 56_676 + 120_028, 'paid_cents': 39_673,
}
display(first['scope_totals'])
display(first['trace'])
display(first['ledger'])

{'count': 3, 'claimed_cents': 15201704, 'paid_cents': 39673}

{'strategy': 'document_checks_then_scoped_sql',
 'portfolio_rows': 20,
 'sql': 'SELECT COUNT(*), COALESCE(SUM(claimed_cents),0), COALESCE(SUM(paid_cents),0) FROM claims WHERE tenant_id = ? AND peril = ?',
 'parameters': ['SYNTHETIC_A', 'water'],
 'workbook_sha256': 'ce2703968c8b1ef2b15f846639b9b15dfcd50a01f3d8cd0c35c9dbed093503e3',
 'database_sha256': '1cd589e95b3d0aa02053af82d55c0d1aaca6108aab2dfcc5e1e05fb0f2e0fec3',
 'source': 'claims.xlsx:Claims',
 'generation': 'deterministic_no_model',
 'scope_origin': 'case metadata matched to ledger; not user-provided SQL'}

{'claim_id': 'SYN-C0000001',
 'policy_id': 'SYN-P0000001',
 'tenant_id': 'SYNTHETIC_A',
 'loss_date': '2025-01-01',
 'peril': 'water',
 'status': 'open',
 'claimed_cents': 15025000,
 'paid_cents': 0,
 'currency': 'EUR',
 'source_name': 'claims.xlsx:Claims',
 'source_row': 2}

## 3. Inspect the actual extracted source text
Text and page numbers were cached from the synthetic PDFs. The cache has source hashes and worksheet row locators. Hash identity does not prove authenticity, legal applicability or semantic correctness. The parser understands the authored field grammar; it is not general document understanding.

In [4]:
for document in first['case']['documents']:
    print('\nSOURCE:', document['filename'], 'SHA-256:', document['sha256'])
    for page in document['pages']:
        print('PAGE', page['page'])
        print(page['text'])


SOURCE: claim_statement.pdf SHA-256: 3c57f72c17fd577398678861021d41f8e1c5b7ea1f049efa47667e5ea58348aa
PAGE 1
SYNTHETIC INSURANCE LAB / NOT AN INSURER DOCUMENT
Synthetic loss declaration
Claim reference: SYN-C0000001
Policy reference: SYN-P0000001
Declared loss date: 2025-01-01
Claimed cents: 15025000
Currency: EUR
Declared circumstances are unverified training data.
Training fixture only. Authenticity and coverage NOT ASSESSED.


SOURCE: inspection_diagram.pdf SHA-256: 9bc46f46b07c36ff4898a1f7de824034c674a755bdaf30977ea17a692266ee68
PAGE 1
SYNTHETIC INSPECTION / NOT A REAL LOSS
Claim reference: SYN-C0000001
Illustrative moisture readings and room layout
KITCHEN
HALL
A: 21%
B: 12%
C: 8%
Synthetic meter readings; units are printed labels, not estimates.
No scale. Diagram cannot establish cause, date, liability or coverage.
Use the stated observations only; do not infer repair costs.


SOURCE: invoice.pdf SHA-256: f4eda4c080f30ebeaa19949a4c12707fdc20adbc7b8092d660450121b1d209a3
PAGE 1
SY

## 4. Use the prepared 100,000-row cache in the UI
The local screen is `/#/claims/analytics`, also linked from the insurance text workbench. Six scenarios expose source text, ledger locators, SQL parameters, scoped totals and explicit non-coverage status. Starting a new analysis clears the previous result.

To prepare a cache once from the repository root:
```powershell
.venv/Scripts/python.exe -m ingestion.prepare_insurance_runtime
```
An existing cache is not overwritten. For a new version, use `--output data/insurance_v2/runtime/version_2`, then set the backend's `INSURANCE_RUNTIME_DIR` to that directory. Runtime data is local and Git-ignored; this notebook does not deploy it.

## Acceptance and remaining work
Passing establishes the narrow end-to-end connection: real PDF/XLSX fixtures → preparation → read-only API → traceable data. It does not establish coverage reasoning, arbitrary uploads, free-form questions, LLM generation, authentication, concurrent production capacity or automatic adaptation to unknown schemas. The browser suite separately exercises desktop, tablet and mobile. No public insurer document or real client record is promoted in this increment.